# Level 4 — Kernel Roofline Analysis (GPT-2 on T4)

Move beyond *where* GPU time goes (Level 3) to **whether each operation is using the GPU efficiently**.

**The roofline model in one sentence:** every kernel is bottlenecked by either compute or memory bandwidth, and a 2D plot of *arithmetic intensity* vs *achieved performance* tells you which.

**What this notebook does:**
1. Defines T4 hardware specs (compute peaks, memory bandwidth, ridge points)
2. Defines the GPT-2 small architecture from Level 3
3. Computes FLOPs, bytes accessed, and arithmetic intensity for each kernel category
4. Uses the measured times from the Level 3 trace to compute achieved performance
5. Plots the roofline with each kernel marked as compute-bound, memory-bound, or under-utilizing
6. Writes up the takeaways

**No GPU needed** — this is a pure analysis built on numbers you already collected in Level 3.


## 1. Hardware specs — the NVIDIA T4


In [ ]:
# T4 hardware reference (NVIDIA Turing TU104)
PEAK_FP32_TFLOPS = 8.1            # CUDA cores at boost clock
PEAK_FP16_TC_TFLOPS = 65.0        # Tensor Cores, FP16
HBM_BANDWIDTH_GBS = 320.0         # GDDR6 memory bandwidth

RIDGE_FP32 = (PEAK_FP32_TFLOPS * 1e12) / (HBM_BANDWIDTH_GBS * 1e9)
RIDGE_FP16 = (PEAK_FP16_TC_TFLOPS * 1e12) / (HBM_BANDWIDTH_GBS * 1e9)

print(f'T4 Peak FP32           : {PEAK_FP32_TFLOPS:6.1f} TFLOPS')
print(f'T4 Peak FP16 (TC)      : {PEAK_FP16_TC_TFLOPS:6.1f} TFLOPS')
print(f'T4 HBM bandwidth       : {HBM_BANDWIDTH_GBS:6.1f} GB/s')
print(f'Ridge point (FP32)     : {RIDGE_FP32:6.1f} FLOPs/byte  <-- below = mem-bound, above = compute-bound')
print(f'Ridge point (FP16 TC)  : {RIDGE_FP16:6.1f} FLOPs/byte')

## 2. Model + workload (from Level 3)

These are the exact dimensions used in the GPT-2 profiling run.


In [ ]:
# GPT-2 small variant from Level 3
N_LAYER = 6
N_HEAD  = 8
D_MODEL = 512
D_HEAD  = D_MODEL // N_HEAD     # = 64
D_FF    = 4 * D_MODEL           # = 2048
B       = 8                     # batch size
S       = 128                   # sequence length
STEPS   = 3                     # active profiled steps
FWDBWD_FACTOR = 3               # forward + backward ~= 3x forward FLOPs

BYTES_PER_FP32 = 4

tokens_per_step = B * S
print(f'Tokens per step           : {tokens_per_step}')
print(f'Layers x heads x head_dim : {N_LAYER} x {N_HEAD} x {D_HEAD}')
print(f'FFN inner dimension       : {D_FF}')

## 3. FLOPs and bytes per kernel category

For each category we estimate:
- **FLOPs** from the standard formulas (matmul = 2 * M * N * K, etc.)
- **Bytes** from the input + output tensor sizes in FP32
- **Arithmetic intensity** = FLOPs / Bytes

We multiply by `N_LAYER * FWDBWD_FACTOR * STEPS` to match the Level 3 profile (3 steps, fwd+bwd).


In [ ]:
from dataclasses import dataclass

@dataclass
class KernelCategory:
    name: str
    flops_per_call: float        # arithmetic per single call (forward, single layer)
    bytes_per_call: float        # memory traffic per single call
    calls_per_step: int          # forward calls per training step (across layers)
    measured_time_ms: float      # from Level 3 trace (across STEPS active steps)

    @property
    def total_flops(self):
        # 1 forward + ~2 backward = 3x; STEPS profiled steps
        return self.flops_per_call * self.calls_per_step * FWDBWD_FACTOR * STEPS

    @property
    def total_bytes(self):
        return self.bytes_per_call * self.calls_per_step * FWDBWD_FACTOR * STEPS

    @property
    def arithmetic_intensity(self):
        return self.flops_per_call / self.bytes_per_call

    @property
    def achieved_tflops(self):
        sec = self.measured_time_ms / 1000.0
        return (self.total_flops / sec) / 1e12 if sec > 0 else 0.0

    def classify(self):
        if self.arithmetic_intensity > RIDGE_FP32:
            return 'Compute-bound (FP32)'
        return 'Memory-bound (FP32)'

# ---- FLOPs/bytes formulas ----

# FFN matmul (largest matmul: [B*S, D] @ [D, FF])
ffn_flops = 2 * (B*S) * D_MODEL * D_FF
ffn_bytes = BYTES_PER_FP32 * ((B*S)*D_MODEL + D_MODEL*D_FF + (B*S)*D_FF)

# Attention QK^T (per layer, summed over heads)
# (B,H,S,d) @ (B,H,d,S) -> (B,H,S,S)
attn_flops = 2 * B * N_HEAD * S * S * D_HEAD
attn_bytes = BYTES_PER_FP32 * (B*N_HEAD*S*D_HEAD + B*N_HEAD*D_HEAD*S + B*N_HEAD*S*S)

# LayerNorm over last dim (B*S tokens, D features)
ln_flops = 8 * (B*S) * D_MODEL                        # mean, var, normalize, scale, shift
ln_bytes = BYTES_PER_FP32 * 2 * (B*S) * D_MODEL       # read input + write output

# Elementwise (e.g. residual add: 2 reads + 1 write)
ew_flops = (B*S) * D_MODEL
ew_bytes = BYTES_PER_FP32 * 3 * (B*S) * D_MODEL

# Times measured in Level 3 (ms across 3 active steps)
kernels = [
    KernelCategory('Matmul (FFN)',       ffn_flops, ffn_bytes, calls_per_step=2*N_LAYER, measured_time_ms=142.65),
    KernelCategory('Attention (QK^T)',   attn_flops, attn_bytes, calls_per_step=N_LAYER,  measured_time_ms=18.06),
    KernelCategory('LayerNorm',          ln_flops, ln_bytes,    calls_per_step=2*N_LAYER, measured_time_ms=4.05),
    KernelCategory('Elementwise',        ew_flops, ew_bytes,    calls_per_step=4*N_LAYER, measured_time_ms=41.53),
]

print(f'{"Kernel":<20} {"AI":>10} {"Achieved":>14} {"% of peak":>10} {"Verdict":<22}')
print('-' * 80)
for k in kernels:
    pct_peak = 100 * k.achieved_tflops / PEAK_FP32_TFLOPS
    print(f'{k.name:<20} {k.arithmetic_intensity:>9.1f}  {k.achieved_tflops:>11.2f} TFLOPS  {pct_peak:>8.1f}%  {k.classify()}')

## 4. The Roofline plot

Each kernel is a point. The two lines are the GPU's hardware limits — kernels can never exceed the roof.

- **Diagonal line (left side)**: memory-bandwidth ceiling. As intensity increases, this rises proportionally.
- **Horizontal line (right side)**: compute ceiling. Once you have enough FLOPs per byte, you saturate the math units.
- **Ridge point**: where they meet. Below it = memory-bound. Above it = compute-bound.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6.5))

ai_range = np.logspace(-2, 4, 200)

# FP32 roof
fp32_roof = np.minimum(HBM_BANDWIDTH_GBS * ai_range, PEAK_FP32_TFLOPS * 1e3)  # GFLOPS
ax.plot(ai_range, fp32_roof, 'k-', linewidth=2, label=f'T4 FP32 roof ({PEAK_FP32_TFLOPS} TFLOPS / {HBM_BANDWIDTH_GBS} GB/s)')

# FP16 tensor core roof (for context)
fp16_roof = np.minimum(HBM_BANDWIDTH_GBS * ai_range, PEAK_FP16_TC_TFLOPS * 1e3)
ax.plot(ai_range, fp16_roof, 'k--', linewidth=1.2, alpha=0.5, label=f'T4 FP16 Tensor Core roof ({PEAK_FP16_TC_TFLOPS} TFLOPS)')

# Ridge points
ax.axvline(RIDGE_FP32, color='gray', linestyle=':', alpha=0.6)
ax.text(RIDGE_FP32*1.1, 12, f'FP32 ridge\n{RIDGE_FP32:.0f} FLOPs/byte', fontsize=9, color='gray')

# Plot each kernel
colors = ['#2563eb', '#dc2626', '#ea580c', '#7c3aed']
for k, color in zip(kernels, colors):
    achieved_gflops = k.achieved_tflops * 1000
    ax.scatter(k.arithmetic_intensity, achieved_gflops, s=180, color=color, zorder=5, edgecolor='white', linewidth=1.5)
    ax.annotate(k.name, (k.arithmetic_intensity, achieved_gflops),
                xytext=(10, 8), textcoords='offset points', fontsize=10, fontweight='bold', color=color)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Arithmetic Intensity (FLOPs / byte)', fontsize=11)
ax.set_ylabel('Performance (GFLOPS)', fontsize=11)
ax.set_title('GPT-2 small kernels on T4 — Roofline analysis', fontsize=13, fontweight='bold')
ax.set_xlim(0.05, 5000)
ax.set_ylim(0.5, 1e5)
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('roofline_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roofline_plot.png')

## 5. What this tells us — the takeaways

**Matmul (FFN) — compute-bound, ~31% of FP32 peak**
Above the ridge by a wide margin (AI ≈ 146 FLOPs/byte vs ridge of 25). The kernel is doing the right amount of arithmetic per byte; it's just not using the most powerful units.
→ *Optimization path:* switch to FP16/BF16 to engage Tensor Cores. Theoretical max becomes 65 TFLOPS instead of 8.1.

**Attention (QK^T) — barely memory-bound at this size**
AI ≈ 21 sits just below the FP32 ridge of 25. At larger sequence lengths the FLOPs grow as O(S²·d) while memory grows as O(S²), so attention shifts toward compute-bound at S=2048 and above. This is also why FlashAttention's tiling matters — it raises attention's effective AI by keeping intermediate matrices in SRAM.

**LayerNorm — severely memory-bound (1.4% of peak)**
AI = 1 FLOP/byte. The kernel is *not* slow because the code is bad — it's slow because every byte loaded only generates one FLOP of work. Speeding up the math wouldn't help.
→ *Optimization path:* kernel fusion. Combining LayerNorm with the next op (e.g. linear projection) lets the data stay in registers/SRAM instead of round-tripping through HBM.

**Elementwise — catastrophically memory-bound**
AI < 0.1 FLOP/byte. Two reads + one write for every single FLOP. The fix is the same: fuse with neighbors. PyTorch 2's `torch.compile` does this automatically via inductor.

**The interview-ready summary:**
> On T4 with this GPT-2 config, matmuls are compute-bound and would benefit most from tensor cores (mixed precision). Attention sits near the ridge — its bottleneck depends on sequence length. LayerNorm and elementwise ops are memory-bound and benefit from fusion, not faster math.


## 6. Optional: re-run with FP16/Tensor Core ridge

If you re-run Level 3 with AMP enabled, here's what the picture would change to:
- New ridge point becomes **203 FLOPs/byte** (FP16 Tensor Cores)
- Matmuls at AI=146 would now sit *below* the ridge → memory-bound
- This is the counterintuitive lesson: **as compute gets faster, more things become memory-bound**
- It's why every modern GPU iteration (Volta → Ampere → Hopper → Blackwell) ships with more memory bandwidth, not just more FLOPs
